In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import mysql.connector
print("Everything is Ready!")

In [ ]:
import sys

print(sys.executable)

In [ ]:
import mysql.connector

print("MySQL connector is working!")

In [ ]:
from getpass import getpass
import mysql.connector

password = getpass("Enter MySQL password: ")

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=password,
    database="ecommerce_de"
)

print("Connected successfully!")

In [ ]:
import pandas as pd

query = """
SELECT *
FROM fact_order_items;
"""

fact = pd.read_sql(query, conn)

fact

In [ ]:
fact.shape

In [ ]:
fact.columns

In [ ]:
fact.head()

In [ ]:
customer_query = "SELECT * FROM dim_customer;"
product_query = "SELECT * FROM dim_product;"
date_query = "SELECT * FROM dim_date;"

customers = pd.read_sql(customer_query, conn)
products = pd.read_sql(product_query, conn)
dates = pd.read_sql(date_query, conn)

In [ ]:
customers.shape
products.shape
dates.shape

In [ ]:
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Dates:", dates.shape)

In [ ]:
fact_customer = fact.merge(
    customers,
    how='left',
    on='customer_key'
)

fact_customer

In [ ]:
fact_product = fact_customer.merge(
    products,
    how='left',
    on='product_key'
)
fact_product

In [ ]:
fact_date = fact_product.merge(
    dates,
    how='left',
    on='date_key'
)
fact_date

In [ ]:
fact_date.columns

In [ ]:
fact_date.head

In [ ]:
total_sales = fact_date['item_amount'].sum()
print("Toal_sales:",total_sales)

In [ ]:
total_orders=fact_date["order_id"].nunique()
print("Total_orders:",total_orders)

In [ ]:
order_totals = fact_date.groupby('order_id')['item_amount'].sum()
average_order_value = order_totals.mean()
print("Average Order Value:", average_order_value)

In [ ]:
sales_by_category = fact_date.groupby('category')['item_amount'].sum()
print("sales_by_category:",sales_by_category)

In [ ]:
sales_by_category.plot(
    kind='bar',
    figsize=(8,5),
    title='Sales by Category',
    ylabel='Sales',
    xlabel='Category'
)

In [ ]:
ax = sales_by_category.plot(
    kind='bar',
    figsize=(8,5),
    title='Sales by Category',
    ylabel='Sales',
    xlabel='Category'
)

ax.bar_label(ax.containers[0])

In [ ]:
ax = sales_by_category.plot(
    kind='bar',
    figsize=(8,5),
    title='Sales by Category',
    ylabel='Sales',
    xlabel='Category'
)

ax.bar_label(
    ax.containers[0],
    fmt=lambda x: f'₹{x:,.0f}'
)

In [ ]:
ax = sales_by_category.plot(
    kind='bar',
    figsize=(8,5),
    title='Sales by Category',
    ylabel='Sales',
    xlabel='Category'
)

ax.bar_label(
    ax.containers[0],
    fmt=lambda x: f'₹{x:,.0f}'
)

ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f'₹{x/1000:.0f}K')
)
plt.tight_layout()
plt.savefig("sales_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fact_date['full_date'].dtype

In [ ]:
fact_date['full_date'] = pd.to_datetime(fact_date['full_date'])

In [ ]:
fact_date['full_date'].dtype

In [ ]:
fact_date['month'] = fact_date['full_date'].dt.month

In [ ]:
monthly_sales = fact_date.groupby('month')['item_amount'].sum()
monthly_sales

In [ ]:
import calendar

monthly_sales.index = monthly_sales.index.map(
    lambda x: calendar.month_name[x]
)
monthly_sales

In [ ]:
ax = monthly_sales.plot(
    kind='line',
    marker='o',
    figsize=(8,5),
    title='Monthly Sales Trend',
    ylabel='Sales',
    xlabel='Month'
)
ax.set_xticks(range(len(monthly_sales.index)))
ax.set_xticklabels(monthly_sales.index)

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f'₹{x/1000:.0f}K')
)

for x, y in enumerate(monthly_sales):
    ax.annotate(
        f'₹{y:,.0f}',
        (x, y),
        xytext=(0, 8),
        textcoords='offset points',
        ha='center'
    )
plt.tight_layout()
plt.savefig("monthly_sales_trend.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
top_customers = fact_date.groupby('customer_name')['item_amount'].sum()
top_customers = top_customers.sort_values(ascending=False)
top_customers

In [ ]:
ax = top_customers.plot(
    kind='bar',
    figsize=(8,5),
    title='Top Customers by Sales',
    ylabel='Sales',
    xlabel='Customer'
)

ax.bar_label(
    ax.containers[0],
    fmt=lambda x: f'₹{x:,.0f}'
)

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f'₹{x/1000:.0f}K')
)
plt.tight_layout()
plt.savefig("top_customers.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
top_products = fact_date.groupby('product_name')['item_amount'].sum()
print("Top_products:",top_products)

In [ ]:
top_products = top_products.sort_values(ascending=False)
print(top_products)

In [ ]:
ax = top_products.plot(
    kind='bar',
    figsize=(8,5),
    title='Top Products by Sales',
    ylabel='Sales',
    xlabel='Product'
)
ax.bar_label(
    ax.containers[0],
    fmt=lambda x: f'₹{x:,.0f}'
)

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f'₹{x/1000:.0f}K')
)
plt.tight_layout()
plt.savefig("top_products.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
total_sales_display = f"₹{total_sales:,.0f}"
print(total_sales_display)

In [ ]:
total_orders_display = f"{total_orders}"
print(total_orders_display)

In [ ]:
average_order_value_display = f"₹{average_order_value:,.0f}"
print(average_order_value_display)

In [ ]:
kpis = {
    "Total Sales": total_sales_display,
    "Total Orders": total_orders_display,
    "Average Order Value": average_order_value_display
}

print(kpis)

In [ ]:
print(kpis["Total Sales"])
print(kpis["Total Orders"])
print(kpis["Average Order Value"])

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(12, 8))

fig.suptitle(
    "E-Commerce Sales Dashboard",
    fontsize=20
)

plt.show()

In [ ]:
import os

files = [
    "sales_by_category.png",
    "monthly_sales_trend.png",
    "top_customers.png",
    "top_products.png"
]

for file in files:
    print(file, "→", os.path.exists(file))